<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [ ]:
%pip install -U -q keras-hub keras

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [ ]:
import keras
import keras_hub

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")

In [ ]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

In [ ]:
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)

print(gemma_lm.generate(prompt, max_length=256))

In [ ]:
!wget -O databricks-dolly-15k.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

In [ ]:
import json

prompts = []
responses = []
line_count = 0

with open("databricks-dolly-15k.jsonl") as file:
    for line in file:
        if line_count >= 1000:
            break  # Limit the training examples, to reduce execution time.

        examples = json.loads(line)
        # Filter out examples with context, to keep it simple.
        if examples["context"]:
            continue
        # Format data into prompts and response lists.
        prompts.append(examples["instruction"])
        responses.append(examples["response"])

        line_count += 1

features = {
    "prompts": prompts,
    "responses": responses
}

In [ ]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

In [ ]:
gemma_lm.summary()

In [ ]:
gemma_lm.backbone.enable_lora(rank=4)

In [ ]:
gemma_lm.summary()

In [ ]:
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

In [ ]:
gemma_lm.fit(features, epochs=1, batch_size=1)

In [ ]:
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)

print(gemma_lm.generate(prompt, max_length=256))